# S6E6 — Colab runner (idempotent)

A **thin, stable launcher.** It only mounts Drive, sets secrets, and **fresh-clones** the
repo — then runs `colab/bootstrap.py`, which holds ALL the run logic (install, data, vendor,
run, dashboard, Drive-sync, diary push) and is pulled fresh every time. **You never edit this
notebook, and you never re-upload anything.**

- **Run an experiment:** Runtime → Run all.
- **Switch experiment / change the flow:** edit `SPRINT_ACTIVE.txt` (or `colab/bootstrap.py`)
  in the repo, **push**, then re-run the **Run** cell — it re-clones and picks up the latest.
- **Prereqs (Colab Secrets 🔑, notebook access ON):**
  `KAGGLE_API_TOKEN` (required — kaggle 2.x bearer token),
  `GH_TOKEN` (optional — fine-grained PAT, Contents:RW, to persist the diary to git).
  Drive holds artifacts (probs/submissions/reports/experiments.jsonl). GPU optional (GBDT runs on CPU).

## 1. Setup — Drive + secrets (stable; rarely changes)

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.environ['DRIVE_ROOT'] = '/content/drive/MyDrive/Colab Notebooks/kaggle/s6e6'
os.makedirs(os.environ['DRIVE_ROOT'], exist_ok=True)

from google.colab import userdata
for name in ('KAGGLE_API_TOKEN', 'GH_TOKEN'):
    try:
        v = userdata.get(name)
        if v:
            os.environ[name] = v; print(f'✓ {name} set')
        else:
            print(f'⚠ {name} empty')
    except Exception:
        req = ' (REQUIRED)' if name == 'KAGGLE_API_TOKEN' else ' (optional)'
        print(f'⚠ no {name}{req} — add via 🔑, notebook access ON')


## 2. Run — fresh clone + bootstrap (all logic lives in the repo)

In [ ]:
%cd /content
!rm -rf playground-s6e6
!git clone -q https://github.com/SirGrigor/playground-s6e6.git
%cd playground-s6e6
!git log -1 --oneline
!python colab/bootstrap.py

## 3. Review — open the HTML dashboard inline

In [ ]:
from pathlib import Path
from IPython.display import HTML, display
p = Path('/content/playground-s6e6/reports/dashboard.html')
display(HTML(p.read_text())) if p.exists() else print('no dashboard.html yet')

## 4. (optional) Download the newest submission

In [ ]:
from pathlib import Path
from google.colab import files
subs = sorted(Path('/content/playground-s6e6/submissions').glob('*.csv'), key=lambda p: -p.stat().st_mtime)
if subs:
    print('newest:', subs[0].name); files.download(str(subs[0]))
else:
    print('no submissions yet')